# FoodWise Reads: Improved Food Book Recommender
Ironhack project covering web scraping, APIs, EDA, TF-IDF, SVD, K-Means, PCA, SQL and Streamlit.

In [ ]:
import json, joblib, sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from src.collect_data import collect_all
from src.train_model import train
from src.database import build_database, example_queries
from src.recommender import recommend_books

## 1. Data collection
The collection script combines 500 scraped cookbook records with 700 balanced food-domain API records. After title-author deduplication, the catalogue contains 1,196 unique books and exceeds the 1,050-book submission target. Run this cell only when a fresh collection is needed.

In [ ]:
# collection_summary = collect_all('data')
# collection_summary

## 2. Cleaning and initial exploration

In [ ]:
books = pd.read_csv('data/books_combined.csv')
print('Shape:', books.shape)
display(books.head())
display(books.isna().mean().sort_values(ascending=False).to_frame('missing_share'))
display(books.groupby('source').size().to_frame('books'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
books['source'].value_counts().plot(kind='bar', ax=axes[0], title='Books by source')
books['source_topic'].value_counts().head(12).sort_values().plot(kind='barh', ax=axes[1], title='Largest food topics')
plt.tight_layout(); plt.show()

## 3. Leakage-controlled NLP and clustering
The modelling catalogue balances six food domains at 100 books each, producing 600 recommendation candidates. TF-IDF uses titles and cleaned subject metadata. Known domain and source-topic labels are held out. Candidate SVD dimensions are compared at k=6, followed by five-seed stability testing.

In [ ]:
# Re-run only when the catalogue or modelling choices change:
# metrics = train('data/books_combined.csv')
metrics = json.loads(Path('models/metrics.json').read_text())
display(pd.Series(metrics, name='result'))
selection = pd.read_csv('outputs/model_selection.csv')
selection.plot(x='components', y='silhouette', marker='o', title='Silhouette by SVD components')
plt.axhline(0.50, color='grey', linestyle=':', label='Target = 0.50')
plt.axvline(metrics['selected_svd_components'], color='darkorange', linestyle='--')
plt.ylabel('silhouette score'); plt.legend(); plt.show()
display(pd.read_csv('outputs/stability_scores.csv'))

## 4. PCA visualization
PCA reduces the selected six-dimensional representation to two dimensions for visualization only. K-Means uses the complete selected representation.

In [ ]:
pca = pd.read_csv('outputs/pca_coordinates.csv')
plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca, x='PC1', y='PC2', hue='known_domain', palette='Set2', s=30)
plt.title('PCA projection of the improved food-book model'); plt.show()

## 5. SQL storage and queries

In [ ]:
build_database()
queries = example_queries()
display(queries['cluster_sizes'])
display(queries['top_rated'])
display(queries['sources'])

## 6. Recommendation demonstration
The function first restricts candidates to the selected book's cluster, then ranks them with cosine similarity.

In [ ]:
clustered = pd.read_csv('data/clustered_books.csv')
matrix = sparse.load_npz('models/book_matrix.npz')
example_title = clustered.iloc[0]['title']
example_author = clustered.iloc[0]['authors']
result = recommend_books(example_title, clustered, matrix, example_author, number=5)
print('Selected:', result['selected'])
display(pd.DataFrame(result['recommendations']))

## 7. Conclusion
The expanded model achieved a silhouette score of 0.6093, ARI of 0.6665 and NMI of 0.6667 across 600 modelled books. The same silhouette was reproduced across five random seeds. The project therefore exceeds both the 1,050-book data target and the 0.50 clustering target.